In [ ]:
import scanpy as sc
import pandas as pd
import numpy as np
import scipy.sparse as sp

In [ ]:
adata_single = sc.read_h5ad("/home/quantori/Downloads/data/joung/joun_perturbed_log1p_grn_subset_1168_genes.h5ad") ## Google Drive: gCAL_data > Joung2013_results

In [ ]:
adata_double = sc.read_h5ad("/home/quantori/Downloads/data/joung/joun_double_perturbed_log1p_grn_subset_1168_genes.h5ad") ## Google Drive: gCAL_data > Joung2013_results

In [4]:
adata_single

AnnData object with n_obs × n_vars = 12176 × 1168
    obs: 'n_genes', 'percent_mito', 'n_counts', 'batch', 'TF', 'condition', 'louvain'
    var: 'n_cells-0-double', 'n_cells-1-double', 'n_cells-2-double', 'n_cells-3-double', 'n_cells-4-double', 'n_cells-5-double', 'n_cells-6-double', 'n_cells-7-double', 'n_cells-8-double', 'highly_variable-double', 'means-double', 'dispersions-double', 'dispersions_norm-double', 'mean-double', 'std-double', 'n_cells-0-0-perturbed', 'n_cells-0-1-0-perturbed', 'n_cells-1-1-0-perturbed', 'n_cells-2-1-0-perturbed', 'n_cells-3-1-0-perturbed', 'n_cells-0-2-0-perturbed', 'n_cells-1-2-0-perturbed', 'n_cells-2-2-0-perturbed', 'n_cells-3-2-0-perturbed', 'n_cells-0-3-0-perturbed', 'n_cells-1-3-0-perturbed', 'n_cells-2-3-0-perturbed', 'n_cells-3-3-0-perturbed', 'n_cells-0-4-0-perturbed', 'n_cells-1-4-0-perturbed', 'n_cells-2-4-0-perturbed', 'n_cells-3-4-0-perturbed', 'n_cells-0-1-perturbed', 'n_cells-1-1-perturbed', 'n_cells-2-1-perturbed', 'n_cells-3-1-perturbed'

In [5]:
adata_double

AnnData object with n_obs × n_vars = 51824 × 1168
    obs: 'n_genes', 'percent_mito', 'n_counts', 'batch', 'TF', 'louvain'
    var: 'n_cells-0', 'n_cells-1', 'n_cells-2', 'n_cells-3', 'n_cells-4', 'n_cells-5', 'n_cells-6', 'n_cells-7', 'n_cells-8'

In [6]:
adata_double.obs['condition'] = adata_double.obs['TF'].str.replace(',', '+', regex=False)

In [7]:
adata_double.obs

,n_genes,percent_mito,n_counts,batch,TF,louvain,condition
"R1.01,R2.01,R3.25,P1.XX-0",963,0.062128,1175.0,0,"TFORF0902-PTF1A,TFORF0098-MSGN1",6,TFORF0902-PTF1A+TFORF0098-MSGN1
"R1.01,R2.18,R3.10,P1.XX-0",532,0.079077,607.0,0,"TFORF0902-PTF1A,TFORF0098-MSGN1",0,TFORF0902-PTF1A+TFORF0098-MSGN1
"R1.01,R2.18,R3.27,P1.XX-0",2226,0.054028,3165.0,0,"TFORF0902-PTF1A,TFORF0098-MSGN1",16,TFORF0902-PTF1A+TFORF0098-MSGN1
"R1.01,R2.18,R3.56,P1.XX-0",2121,0.024637,3166.0,0,"TFORF0902-PTF1A,TFORF0098-MSGN1",11,TFORF0902-PTF1A+TFORF0098-MSGN1
"R1.01,R2.25,R3.28,P1.XX-0",1671,0.039947,2278.0,0,"TFORF0098-MSGN1,TFORF1447-NR5A2",0,TFORF0098-MSGN1+TFORF1447-NR5A2
...,...,...,...,...,...,...,...
"R1.96,R2.86,R3.20,P1.31-8",1240,0.014785,1488.0,8,"TFORF0391-FERD3L,TFORF0098-MSGN1",1,TFORF0391-FERD3L+TFORF0098-MSGN1
"R1.96,R2.87,R3.15,P1.31-8",1385,0.038373,1746.0,8,"TFORF0391-FERD3L,TFORF0098-MSGN1",1,TFORF0391-FERD3L+TFORF0098-MSGN1
"R1.96,R2.94,R3.85,P1.31-8",3540,0.010881,6893.0,8,"TFORF1216-NHLH1,TFORF0098-MSGN1",3,TFORF1216-NHLH1+TFORF0098-MSGN1
"R1.96,R2.94,R3.94,P1.31-8",4196,0.006881,9446.0,8,"TFORF0391-FERD3L,TFORF0098-MSGN1",3,TFORF0391-FERD3L+TFORF0098-MSGN1


In [8]:

def check_preprocessing(adata, sample_n=1000):
    print("Has raw:", adata.raw is not None)
    print("Layers:", list(adata.layers.keys()))
    
    X = adata.X
    
    # Convert small sample to dense if it's sparse
    if sp.issparse(X):
        X_sample = X[:sample_n, :sample_n].toarray()
    else:
        X_sample = np.array(X[:sample_n, :sample_n])
    
    vals = X_sample.ravel()
    print(f"X mean={vals.mean():.3f}, min={vals.min():.3f}, max={vals.max():.3f}")
    
    # Heuristic classification
    if np.all(vals >= 0) and np.all(vals < 15):
        print("→ Likely log1p-normalized data.")
    elif np.any(vals < 0) and np.all(vals < 10):
        print("→ Likely scaled (zero-centered) data.")
    elif np.allclose(vals, vals.astype(int), atol=1e-6):
        print("→ Likely raw counts.")
    else:
        print("→ Mixed or uncertain — check layers/raw.")



In [9]:
check_preprocessing(adata_single)

Has raw: False
Layers: []
X mean=0.010, min=0.000, max=0.382
→ Likely log1p-normalized data.


In [10]:
check_preprocessing(adata_double)

Has raw: False
Layers: []
X mean=0.182, min=0.000, max=5.867
→ Likely log1p-normalized data.


In [11]:
adata_single

AnnData object with n_obs × n_vars = 12176 × 1168
    obs: 'n_genes', 'percent_mito', 'n_counts', 'batch', 'TF', 'condition', 'louvain'
    var: 'n_cells-0-double', 'n_cells-1-double', 'n_cells-2-double', 'n_cells-3-double', 'n_cells-4-double', 'n_cells-5-double', 'n_cells-6-double', 'n_cells-7-double', 'n_cells-8-double', 'highly_variable-double', 'means-double', 'dispersions-double', 'dispersions_norm-double', 'mean-double', 'std-double', 'n_cells-0-0-perturbed', 'n_cells-0-1-0-perturbed', 'n_cells-1-1-0-perturbed', 'n_cells-2-1-0-perturbed', 'n_cells-3-1-0-perturbed', 'n_cells-0-2-0-perturbed', 'n_cells-1-2-0-perturbed', 'n_cells-2-2-0-perturbed', 'n_cells-3-2-0-perturbed', 'n_cells-0-3-0-perturbed', 'n_cells-1-3-0-perturbed', 'n_cells-2-3-0-perturbed', 'n_cells-3-3-0-perturbed', 'n_cells-0-4-0-perturbed', 'n_cells-1-4-0-perturbed', 'n_cells-2-4-0-perturbed', 'n_cells-3-4-0-perturbed', 'n_cells-0-1-perturbed', 'n_cells-1-1-perturbed', 'n_cells-2-1-perturbed', 'n_cells-3-1-perturbed'

In [12]:
adata_double

AnnData object with n_obs × n_vars = 51824 × 1168
    obs: 'n_genes', 'percent_mito', 'n_counts', 'batch', 'TF', 'louvain', 'condition'
    var: 'n_cells-0', 'n_cells-1', 'n_cells-2', 'n_cells-3', 'n_cells-4', 'n_cells-5', 'n_cells-6', 'n_cells-7', 'n_cells-8'

In [13]:
print(adata_single.shape, adata_double.shape)
print(len(set(adata_single.var_names) & set(adata_double.var_names)), "common genes")

# --- Align genes to the same order (intersection) ---
common_genes = adata_single.var_names.intersection(adata_double.var_names)
adata_single = adata_single[:, common_genes]
adata_double = adata_double[:, common_genes]

# --- Concatenate (rows = cells, columns = genes) ---
adata_merged = adata_single.concatenate(
    adata_double,
    join='inner',          # ensure same genes
    batch_key='dataset',   # new column in .obs indicating origin
    batch_categories=['single', 'double']
)

print(adata_merged)
print(adata_merged.obs['dataset'].value_counts())

(12176, 1168) (51824, 1168)
1168 common genes


/tmp/ipykernel_37395/2555912793.py:10: FutureWarning: Use anndata.concat instead of AnnData.concatenate, AnnData.concatenate is deprecated and will be removed in the future. See the tutorial for concat at: https://anndata.readthedocs.io/en/latest/concatenation.html
  adata_merged = adata_single.concatenate(


AnnData object with n_obs × n_vars = 64000 × 1168
    obs: 'n_genes', 'percent_mito', 'n_counts', 'batch', 'TF', 'condition', 'louvain', 'dataset'
    var: 'n_cells-0-double', 'n_cells-1-double', 'n_cells-2-double', 'n_cells-3-double', 'n_cells-4-double', 'n_cells-5-double', 'n_cells-6-double', 'n_cells-7-double', 'n_cells-8-double', 'n_cells-0-double-single', 'n_cells-1-double-single', 'n_cells-2-double-single', 'n_cells-3-double-single', 'n_cells-4-double-single', 'n_cells-5-double-single', 'n_cells-6-double-single', 'n_cells-7-double-single', 'n_cells-8-double-single', 'highly_variable-double-single', 'means-double-single', 'dispersions-double-single', 'dispersions_norm-double-single', 'mean-double-single', 'std-double-single', 'n_cells-0-0-perturbed-single', 'n_cells-0-1-0-perturbed-single', 'n_cells-1-1-0-perturbed-single', 'n_cells-2-1-0-perturbed-single', 'n_cells-3-1-0-perturbed-single', 'n_cells-0-2-0-perturbed-single', 'n_cells-1-2-0-perturbed-single', 'n_cells-2-2-0-perturbe

In [14]:
adata_merged

AnnData object with n_obs × n_vars = 64000 × 1168
    obs: 'n_genes', 'percent_mito', 'n_counts', 'batch', 'TF', 'condition', 'louvain', 'dataset'
    var: 'n_cells-0-double', 'n_cells-1-double', 'n_cells-2-double', 'n_cells-3-double', 'n_cells-4-double', 'n_cells-5-double', 'n_cells-6-double', 'n_cells-7-double', 'n_cells-8-double', 'n_cells-0-double-single', 'n_cells-1-double-single', 'n_cells-2-double-single', 'n_cells-3-double-single', 'n_cells-4-double-single', 'n_cells-5-double-single', 'n_cells-6-double-single', 'n_cells-7-double-single', 'n_cells-8-double-single', 'highly_variable-double-single', 'means-double-single', 'dispersions-double-single', 'dispersions_norm-double-single', 'mean-double-single', 'std-double-single', 'n_cells-0-0-perturbed-single', 'n_cells-0-1-0-perturbed-single', 'n_cells-1-1-0-perturbed-single', 'n_cells-2-1-0-perturbed-single', 'n_cells-3-1-0-perturbed-single', 'n_cells-0-2-0-perturbed-single', 'n_cells-1-2-0-perturbed-single', 'n_cells-2-2-0-perturbe

In [15]:
# Load the TF→community table
# Expecting columns like: gene / tf / TF / symbol  and  community / community_id
# Adjust the 'possible_*' lists below if headers differ.
df = pd.read_csv("/home/quantori/Downloads/data/joung/tf_communities_step_30pct.csv")

possible_gene_cols = ["gene", "tf", "TF", "symbol", "gene_symbol", "Gene", "TF_name"]
possible_comm_cols = ["community", "community_id", "comm", "cluster", "Community"]

gene_col = next((c for c in possible_gene_cols if c in df.columns), None)
comm_col = next((c for c in possible_comm_cols if c in df.columns), None)
if gene_col is None or comm_col is None:
    raise ValueError(
        f"Could not find gene/community columns. "
        f"CSV columns found: {list(df.columns)}. "
        f"Set 'gene_col' and 'comm_col' manually."
    )


# Normalize symbols (strip/upper)
df = df[[gene_col, comm_col]].copy()
df[gene_col] = df[gene_col].astype(str).str.strip()
# If symbols are case-sensitive, remove the .str.upper() lines
df[gene_col] = df[gene_col].str.upper()

# Decide which column in .var holds gene symbols
# Use 'gene_symbols' if present; fall back to var_names
var_gene_col = "gene_symbols" if "gene_symbols" in adata_merged.var.columns else None

if var_gene_col is None:
    # Create a working column from var_names
    adata_merged.var["__genes_for_map__"] = adata_merged.var_names.astype(str)
    var_gene_col = "__genes_for_map__"

# Normalize AnnData symbols the same way as CSV
adata_merged.var[var_gene_col] = adata_merged.var[var_gene_col].astype(str).str.strip()
adata_merged.var[var_gene_col] = adata_merged.var[var_gene_col].str.upper()

# Build a mapping TF → community
tf_to_comm = df.dropna(subset=[gene_col, comm_col]).drop_duplicates(subset=[gene_col]).set_index(gene_col)[comm_col].to_dict()

# Initialize outputs: community and kind
adata_merged.var["community"] = np.nan        # unknown by default
adata_merged.var["kind"] = "TG"               # non-TFs are TG by default

# Assign communities to matched TFs
# Map communities into a new series aligned to .var
mapped_comm = adata_merged.var[var_gene_col].map(tf_to_comm)

# Assign community where we have a mapping
adata_merged.var.loc[mapped_comm.notna(), "community"] = mapped_comm[mapped_comm.notna()].values
adata_merged.var.loc[mapped_comm.notna(), "kind"] = "TF"

# Enforce “TFs must have a valid community”
# Check that every TF in CSV is present in the AnnData and got a community
csv_tfs = set(tf_to_comm.keys())
adata_symbols = set(adata_merged.var[var_gene_col].unique())
missing_in_adata = sorted(csv_tfs - adata_symbols)
if missing_in_adata:
    print(f"Warning: {len(missing_in_adata)} TFs from CSV not found in adata_merged.var[{var_gene_col}]. "
          f"Examples: {missing_in_adata[:10]}")


# Remove helper column if we created it
if "__genes_for_map__" in adata_merged.var.columns:
    adata_merged.var.drop(columns=["__genes_for_map__"], inplace=True)

adata_merged.var["community"] = pd.to_numeric(adata_merged.var["community"], errors="coerce").astype("Int64")
adata_merged.var["community"] = pd.Categorical(adata_merged.var["community"])

print(adata_merged.var["kind"].value_counts(dropna=False))
print(adata_merged.var[["kind", "community"]].head())

kind
TG    1132
TF      36
Name: count, dtype: int64
        kind community
A1CF      TG       NaN
A3GALT2   TG       NaN
AACSP1    TG       NaN
ABCA10    TG       NaN
ABCA7     TG       NaN


In [16]:
adata_merged.var

,n_cells-0-double,n_cells-1-double,n_cells-2-double,n_cells-3-double,n_cells-4-double,n_cells-5-double,n_cells-6-double,n_cells-7-double,n_cells-8-double,n_cells-0-double-single,...,n_cells-1-4-0-perturbed-single,n_cells-2-4-0-perturbed-single,n_cells-3-4-0-perturbed-single,n_cells-0-1-perturbed-single,n_cells-1-1-perturbed-single,n_cells-2-1-perturbed-single,n_cells-3-1-perturbed-single,gene_names-perturbed-single,community,kind
A1CF,508.0,441.0,140.0,161.0,203.0,208.0,163.0,456.0,547.0,508.0,...,52.0,468.0,346.0,265.0,264.0,287.0,286.0,A1CF,NaN,TG
A3GALT2,146.0,124.0,85.0,91.0,108.0,105.0,80.0,205.0,333.0,146.0,...,19.0,107.0,79.0,107.0,106.0,115.0,113.0,A3GALT2,NaN,TG
AACSP1,358.0,236.0,30.0,70.0,34.0,95.0,33.0,121.0,89.0,358.0,...,43.0,360.0,287.0,232.0,232.0,229.0,249.0,AACSP1,NaN,TG
ABCA10,1721.0,1249.0,1185.0,1184.0,1626.0,1326.0,1253.0,2993.0,3829.0,1721.0,...,113.0,766.0,536.0,446.0,429.0,520.0,492.0,ABCA10,NaN,TG
ABCA7,631.0,502.0,277.0,528.0,360.0,513.0,218.0,1467.0,1077.0,631.0,...,458.0,2895.0,1911.0,2380.0,2189.0,2235.0,2423.0,ABCA7,NaN,TG
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
ZNF521,10632.0,8064.0,7517.0,8531.0,8422.0,9213.0,6862.0,19255.0,17180.0,10632.0,...,4578.0,30513.0,22184.0,21001.0,19902.0,22144.0,21121.0,ZNF521,NaN,TG
ZNF540,220.0,176.0,65.0,120.0,114.0,132.0,85.0,311.0,358.0,220.0,...,48.0,419.0,283.0,292.0,257.0,292.0,265.0,ZNF540,NaN,TG
ZNF560,442.0,325.0,68.0,168.0,81.0,198.0,78.0,304.0,247.0,442.0,...,48.0,290.0,205.0,209.0,207.0,193.0,216.0,ZNF560,NaN,TG
ZNF75D,2668.0,2240.0,1768.0,2247.0,2285.0,2439.0,1687.0,5661.0,5901.0,2668.0,...,2940.0,19972.0,13705.0,11173.0,10463.0,11678.0,11146.0,ZNF75D,3,TF


In [17]:
adata_merged.obs["condition"]

R1.21,R2.72,R3.22,P1.62-3-1-perturbed-single                     TFORF0098-MSGN1
R1.56,R2.96,R3.91,P1.30-1-1-perturbed-single                     TFORF0098-MSGN1
R1.02,R2.87,R3.94,P1.30-1-1-perturbed-single                     TFORF0098-MSGN1
R1.48,R2.75,R3.48,P1.62-3-1-perturbed-single                     TFORF0098-MSGN1
R1.28,R2.02,R3.44,P1.62-3-1-perturbed-single                     TFORF0098-MSGN1
                                                              ...               
R1.96,R2.86,R3.20,P1.31-8-double                TFORF0391-FERD3L+TFORF0098-MSGN1
R1.96,R2.87,R3.15,P1.31-8-double                TFORF0391-FERD3L+TFORF0098-MSGN1
R1.96,R2.94,R3.85,P1.31-8-double                 TFORF1216-NHLH1+TFORF0098-MSGN1
R1.96,R2.94,R3.94,P1.31-8-double                TFORF0391-FERD3L+TFORF0098-MSGN1
R1.96,R2.96,R3.93,P1.31-8-double                 TFORF0902-PTF1A+TFORF0098-MSGN1
Name: condition, Length: 64000, dtype: object

In [18]:
def prepare_obs_columns(
    adata,
    condition_col="condition",
    cell_type_dest="cell_type",
    cell_type_candidates=("cell_type","celltype","CellType","cell_labels","celltype_final"),
    var_symbol_col=None,          # if None or missing -> use var_names
    tf_flag_col="kind",           # expected "TF" for TFs
    tf_label="TF",
    ctrl_tokens=("ctrl","control","unperturbed"),
    strip_prefix=True,            # <- remove prefixes like 'TFORF####-'
):
    # symbol space for TF validation
    have_symbol_col = var_symbol_col is not None and var_symbol_col in adata.var.columns
    if have_symbol_col:
        sym_series = adata.var[var_symbol_col].astype(str)
    else:
        sym_series = pd.Series(adata.var_names.astype(str), index=adata.var_names, name="__var_names__")
        var_symbol_col = "__var_names__"
    gene_index = pd.Index(sym_series.values)
    gene_index_set = set(gene_index)

    # TF membership
    if tf_flag_col in adata.var.columns:
        tf_mask = (adata.var[tf_flag_col] == tf_label)
        tf_symbols = set(sym_series[tf_mask].astype(str))
    else:
        tf_symbols = set(gene_index)

    ctrl_norm = {str(x).strip().lower() for x in ctrl_tokens}
    def clean_tf_token(tok: str) -> str:
        t = tok.strip().upper()
        if not strip_prefix:
            return t
        if "-" in t:
            t2 = t.split("-")[-1]
            return t2 or t
        return t

    # ensure cell_type
    if cell_type_dest not in adata.obs.columns:
        src = next((c for c in cell_type_candidates if c in adata.obs.columns), None)
        adata.obs[cell_type_dest] = adata.obs[src].astype(str) if src else "unknown"
    adata.obs[cell_type_dest] = pd.Categorical(adata.obs[cell_type_dest])

    # source for intervention normalization 
    if "intervention" in adata.obs.columns:
        src_series = adata.obs["intervention"].astype(str)
    else:
        if condition_col not in adata.obs.columns:
            raise KeyError(f"Neither 'intervention' nor '{condition_col}' exists in adata.obs.")
        src_series = adata.obs[condition_col].astype(str)

    # normalization rules
    def normalize_intervention(label: str) -> str:
        lab = label.strip()
        if lab.lower() in ctrl_norm:
            return "unperturbed"
        parts = [p.strip() for p in lab.split("+") if p.strip()]
        parts = [p for p in parts if p.lower() not in ctrl_norm]
        parts = [clean_tf_token(p) for p in parts]
        if not parts:
            return "unperturbed"
        uniq_sorted = sorted(set(parts))
        return uniq_sorted[0] if len(uniq_sorted) == 1 else "+".join(uniq_sorted)

    adata.obs["intervention"] = src_series.map(normalize_intervention)
    adata.obs["intervention"] = pd.Categorical(adata.obs["intervention"])

    # QC: interventions reference real TFs
    unique_iv = pd.Index(adata.obs["intervention"].unique())
    non_ctrl = [x for x in unique_iv if x != "unperturbed"]

    missing_genes, non_tf_genes = set(), set()
    for iv in non_ctrl:
        for tf in iv.split("+"):
            if tf not in gene_index_set:
                missing_genes.add(tf)
            if tf not in tf_symbols and tf_flag_col in adata.var.columns:
                non_tf_genes.add(tf)

    if missing_genes:
        print(f"[QC] WARNING: {len(missing_genes)} TF(s) not found in gene index "
              f"({'var['+var_symbol_col+']' if have_symbol_col else 'var_names'}). "
              f"Examples: {sorted(list(missing_genes))[:10]}")
    if non_tf_genes:
        print(f"[QC] WARNING: {len(non_tf_genes)} intervention gene(s) are not labeled as TF in var['{tf_flag_col}']. "
              f"Examples: {sorted(list(non_tf_genes))[:10]}")

    # set: training/test
    adata.obs["set"] = np.where(adata.obs["intervention"] == "unperturbed", "training", "test")
    adata.obs["set"] = pd.Categorical(adata.obs["set"], categories=["training","test"])

    # training must contain controls for each cell_type
    training_ct_counts = adata.obs.loc[adata.obs["set"]=="training", cell_type_dest].value_counts()
    if training_ct_counts.empty:
        print("[QC] WARNING: No 'training' (unperturbed) cells found.")
    else:
        all_cts = adata.obs[cell_type_dest].value_counts().index
        missing_cts = [ct for ct in all_cts if training_ct_counts.get(ct, 0) == 0]
        if missing_cts:
            print(f"[QC] WARNING: cell types with no 'training' cells: {missing_cts[:10]}")

    print("=== Summary ===")
    print("cell_type categories:", list(adata.obs[cell_type_dest].cat.categories))
    print("intervention examples:\n", adata.obs["intervention"].value_counts().head(8))
    print("set counts:\n", adata.obs["set"].value_counts())
    return adata

In [19]:
prepare_obs_columns(adata_merged, condition_col="condition")


[QC] WARNING: 4 TF(s) not found in gene index (var_names). Examples: ['KLF4', 'MSGN1', 'NHLH1', 'PTF1A']
[QC] WARNING: 5 intervention gene(s) are not labeled as TF in var['kind']. Examples: ['FERD3L', 'KLF4', 'MSGN1', 'NHLH1', 'PTF1A']
=== Summary ===
cell_type categories: ['unknown']
intervention examples:
 intervention
FERD3L+MSGN1    23171
MSGN1+PTF1A     15878
PTF1A            4000
unperturbed      4000
FERD3L+FLI1      3490
MSGN1+NHLH1      2653
FERD3L           2472
FLI1+PTF1A       1870
Name: count, dtype: int64
set counts:
 set
test        60000
training     4000
Name: count, dtype: int64


AnnData object with n_obs × n_vars = 64000 × 1168
    obs: 'n_genes', 'percent_mito', 'n_counts', 'batch', 'TF', 'condition', 'louvain', 'dataset', 'cell_type', 'intervention', 'set'
    var: 'n_cells-0-double', 'n_cells-1-double', 'n_cells-2-double', 'n_cells-3-double', 'n_cells-4-double', 'n_cells-5-double', 'n_cells-6-double', 'n_cells-7-double', 'n_cells-8-double', 'n_cells-0-double-single', 'n_cells-1-double-single', 'n_cells-2-double-single', 'n_cells-3-double-single', 'n_cells-4-double-single', 'n_cells-5-double-single', 'n_cells-6-double-single', 'n_cells-7-double-single', 'n_cells-8-double-single', 'highly_variable-double-single', 'means-double-single', 'dispersions-double-single', 'dispersions_norm-double-single', 'mean-double-single', 'std-double-single', 'n_cells-0-0-perturbed-single', 'n_cells-0-1-0-perturbed-single', 'n_cells-1-1-0-perturbed-single', 'n_cells-2-1-0-perturbed-single', 'n_cells-3-1-0-perturbed-single', 'n_cells-0-2-0-perturbed-single', 'n_cells-1-2-0-pertur

In [20]:
s = adata_merged.obs["intervention"].astype(str)

def n_tfs(iv: str) -> int:
    if iv == "unperturbed":
        return 0
    return len(iv.split("+"))

num_tfs = s.map(n_tfs)

n_unpert = (num_tfs == 0).sum()
n_single  = (num_tfs == 1).sum()
n_double  = (num_tfs == 2).sum()

print(f"Unperturbed: {n_unpert}")
print(f"Single TF:  {n_single}")
print(f"Double TF:  {n_double}")

#  which TFs / TF pairs and how many
singles_counts = s[num_tfs == 1].value_counts()
doubles_counts = s[num_tfs == 2].value_counts()

print("\nTop singles:\n", singles_counts.head(10))
print("\nTop doubles:\n", doubles_counts.head(10))

Unperturbed: 4000
Single TF:  8176
Double TF:  51824

Top singles:
 intervention
PTF1A     4000
FERD3L    2472
MSGN1      959
KLF4       325
NHLH1      122
FLI1       119
PAX5        86
NR5A2       74
HNF4A       19
Name: count, dtype: int64

Top doubles:
 intervention
FERD3L+MSGN1    23171
MSGN1+PTF1A     15878
FERD3L+FLI1      3490
MSGN1+NHLH1      2653
FLI1+PTF1A       1870
MSGN1+NR5A2      1302
FLI1+MSGN1        946
FLI1+NHLH1        630
FERD3L+PAX5       324
FLI1+NR5A2        303
Name: count, dtype: int64


In [21]:
adata_merged.write("/home/quantori/Downloads/data/joung/joung_preprocessed_log1p_data.h5ad")
